In [38]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Tải dữ liệu
df = pd.read_csv('Agri_Data_Cleaned.csv')

# 2. Tách các cột số (Trừ cột mục tiêu 'Yield' ra nếu bạn muốn giữ nguyên target)
# Lưu ý: Với hồi quy, thường ta chỉ chuẩn hóa Input (X), không nhất thiết chuẩn hóa Output (y)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
if 'Yield' in numeric_cols:
    numeric_cols.remove('Yield')

# 3. Khởi tạo bộ chuẩn hóa Z-score
scaler = StandardScaler()

# 4. Thực hiện chuẩn hóa
# Tạo dataframe mới để không ảnh hưởng dữ liệu gốc
df_scaled = df.copy()
df_scaled[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# 5. Kiểm tra kết quả
# Ví dụ: Production cũ là 200 (nhỏ) -> Z-score là -0.22
#        Production cũ là 268754 (lớn) -> Z-score là 3.67
#        -> Thứ tự vẫn được bảo toàn!
print(df_scaled[['Production', 'Avg Temp', 'NDVI_Season_Mean']].head())

# Lưu file kết quả
df_scaled.to_csv('Agri_Data_Normalized.csv', index=False)

   Production  Avg Temp  NDVI_Season_Mean
0   -0.228667 -1.324513          1.301556
1   -0.217816 -0.738050          1.301556
2    3.677861  0.434877          1.301556
3   -0.199574 -0.738050          1.301556
4   -0.203749  1.607804         -1.003848


# Xóa cột AP Ratio, Production , Avg Temp,Avg Humidity,Rain_Temp_Ratio, CN_Ratio, sm_surface

In [39]:
import pandas as pd

# 1. Đọc file dữ liệu
file_path = 'Agri_Data_Normalized.csv'
df = pd.read_csv(file_path)

# 2. Kiểm tra và xóa các cột: 'AP Ratio', 'Production', 'Avg Temp', 'Avg Humidity'
# Danh sách các cột muốn xóa
cols_to_drop = ['AP Ratio', 'Production','Avg Temp','Avg Humidity','Is_Extreme_Heat','is_extreme_Heat_Stress_Days','NDVI_Season_Mean','NDVI_Season_Std','FPAR','NDVI_Season_Min']

# Chỉ xóa những cột thực sự tồn tại trong file để tránh lỗi nếu cột không có
existing_cols = [col for col in cols_to_drop if col in df.columns]

if existing_cols:
    df = df.drop(columns=existing_cols)
    print(f"Đã xóa các cột: {existing_cols}")
else:
    print("Không tìm thấy các cột cần xóa trong file.")

# 3. Lưu kết quả ra file mới (để giữ nguyên file gốc)
output_file = 'Agri_Data_Cleaned_Processed.csv'
df.to_csv(output_file, index=False)

print(f"File đã được lưu tại: {output_file}")

# Hiển thị 5 dòng đầu để kiểm tra
print(df.head())

Đã xóa các cột: ['AP Ratio', 'Production', 'Avg Temp', 'Avg Humidity', 'Is_Extreme_Heat', 'is_extreme_Heat_Stress_Days', 'NDVI_Season_Mean', 'NDVI_Season_Std', 'FPAR', 'NDVI_Season_Min']
File đã được lưu tại: Agri_Data_Cleaned_Processed.csv
       Area  District    Season     Crop Name Transplant        Growth  \
0 -0.195962  Bagerhat      Rabi         Wheat   December  Jan To March   
1 -0.192295  Bagerhat      Rabi       Maize 2   December  Jan To March   
2  3.090006  Bagerhat      Rabi          Boro   November  Dec To March   
3 -0.187633  Bagerhat      Rabi  Sweet Potato   November  Dec To March   
4 -0.193155  Bagerhat  Kharif 1         Mango      April  April To May   

         Harvest  Max Temp  Min Temp  Max Relative Humidity  ...  \
0          April -1.843403 -0.287335              -1.288481  ...   
1          April -0.656067 -0.564733              -1.288481  ...   
2          April  1.548984 -0.934598              -0.298672  ...   
3          April -1.164925 -0.009937      

In [40]:
import pandas as pd
import numpy as np

# 1. Tải dữ liệu
df = pd.read_csv('Agri_Data_Final_Optimized.csv')

# 2. Xử lý ngoại lai (Outlier)
# Loại bỏ giá trị Yield > 900 (trường hợp Jack Fruit bất thường)
df_clean = df[df['Yield'] < 900].copy()
print(f"Đã loại bỏ {len(df) - len(df_clean)} dòng ngoại lai.")

# 3. Chuẩn hóa theo từng loại cây (Method 2: Z-score) và GHI ĐÈ cột Yield cũ
def calculate_zscore(x):
    std = x.std()
    if std == 0:
        return 0  # Nếu chỉ có 1 giá trị hoặc tất cả giống nhau thì Z=0
    return (x - x.mean()) / std

# Áp dụng hàm transform và gán ngược lại vào cột 'Yield'
df_clean['Yield'] = df_clean.groupby('Crop Name')['Yield'].transform(calculate_zscore)

# 4. Kiểm tra và Lưu file
print("\nVí dụ kết quả chuẩn hóa (Cột Yield đã được ghi đè):")
print(df_clean[df_clean['Crop Name'] == 'Jack Fruit'][['Crop Name', 'Yield']].sort_values(by='Yield', ascending=False).head())

# Lưu thành file mới
output_filename = 'Agri_Data_Yield_Standardized.csv'
df_clean.to_csv(output_filename, index=False)
print(f"\nĐã lưu file xử lý xong: {output_filename}")

Đã loại bỏ 0 dòng ngoại lai.

Ví dụ kết quả chuẩn hóa (Cột Yield đã được ghi đè):
       Crop Name     Yield
1363  Jack Fruit  3.933195
1962  Jack Fruit  3.754846
3857  Jack Fruit  2.549397
3006  Jack Fruit  2.347684
3790  Jack Fruit  1.739618

Đã lưu file xử lý xong: Agri_Data_Yield_Standardized.csv


In [41]:
import pandas as pd

# 1. Tải dữ liệu
filename = 'Agri_Data_Yield_Standardized.csv'  # Tên file vừa tạo ở bước trước
df = pd.read_csv(filename)

# 2. Tạo danh sách các hậu tố cần xóa
suffixes_to_remove = ['_x_Crop Name_Optimized', '_x_District_Optimized']

# 3. Duyệt qua các cột và đổi tên
rename_mapping = {}
for col in df.columns:
    for suffix in suffixes_to_remove:
        if col.endswith(suffix):
            # Tạo tên mới bằng cách xóa hậu tố
            new_name = col.replace(suffix, '')
            rename_mapping[col] = new_name
            break  # Đã tìm thấy và xử lý xong cột này, chuyển sang cột tiếp theo

# Thực hiện đổi tên
if rename_mapping:
    df.rename(columns=rename_mapping, inplace=True)
    print(f"Đã đổi tên thành công {len(rename_mapping)} cột.")
    # In ra một vài ví dụ để kiểm tra
    print("Ví dụ:")
    for old, new in list(rename_mapping.items())[:5]:
        print(f"  - {old}  ->  {new}")
else:
    print("Không tìm thấy cột nào có hậu tố cần xóa.")

# 4. Lưu lại file (ghi đè hoặc tạo file mới tùy bạn chọn tên file)
output_filename = 'Agri_Data_Final_Ready_For_Model.csv'
df.to_csv(output_filename, index=False)
print(f"\nĐã lưu file hoàn chỉnh: {output_filename}")

Đã đổi tên thành công 18 cột.
Ví dụ:
  - Max Temp_x_District_Optimized  ->  Max Temp
  - Min Temp_x_District_Optimized  ->  Min Temp
  - Min Relative Humidity_x_District_Optimized  ->  Min Relative Humidity
  - NDVI_Season_Max_x_Crop Name_Optimized  ->  NDVI_Season_Max
  - NDVI_Season_CV_x_Crop Name_Optimized  ->  NDVI_Season_CV

Đã lưu file hoàn chỉnh: Agri_Data_Final_Ready_For_Model.csv


In [42]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
def super_optimize_features(file_path, target_col='Yield'):
    print(f"--- BẮT ĐẦU TỐI ƯU HÓA SÂU: {file_path} ---")
    
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file '{file_path}'")
        return

    # 1. Xác định các cột vẫn còn yếu (tương quan < 0.05)
    # Chúng ta lọc ra các cột số để kiểm tra
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if target_col in numeric_cols: numeric_cols.remove(target_col)
    
    # Tính tương quan hiện tại
    current_corrs = df[numeric_cols].corrwith(df[target_col]).abs()
    
    # Lấy danh sách các cột yếu (dưới 0.05)
    weak_cols = current_corrs[current_corrs < 0.05].index.tolist()
    
    print(f"Phát hiện {len(weak_cols)} cột yếu cần xử lý lại: {weak_cols}")
    
    # Danh sách các "chìa khóa" để thử ghép (Grouping Keys)
    # Đây là các biến phân loại có thể gây ra hiệu ứng Simpson
    potential_keys = ['Crop Name', 'Season', 'District', 'Dominant_Soil_Texture', 'Water_Availability_Cat']
    # Chỉ giữ lại các key thực sự có trong dataframe
    grouping_keys = [k for k in potential_keys if k in df.columns]
    
    results_log = []
    cols_to_drop = []

    # 2. Vòng lặp "Thử sai" (Trial & Error) cho từng cột yếu
    for col in weak_cols:
        best_corr = current_corrs[col]
        best_key = None
        best_new_col_name = None
        best_slope_map = None
        
        # Thử ghép cột yếu này với từng key (Season, District, Soil, v.v.)
        for key in grouping_keys:
            # Bỏ qua nếu cột đó chính là key (hoặc tương tự)
            if col == key: continue
            
            # Tính Slope cho từng nhóm của key này
            slopes = {}
            
            groups = df[key].unique()
            for group in groups:
                subset = df[df[key] == group]
                # Chỉ tính nếu nhóm có đủ dữ liệu (>5 mẫu)
                if len(subset) > 5:
                    # Nếu cột này là hằng số trong nhóm (độ lệch chuẩn = 0) -> Slope = 0
                    if subset[col].std() == 0: 
                        slopes[group] = 0
                    else:
                        model = LinearRegression()
                        model.fit(subset[[col]], subset[target_col])
                        slopes[group] = model.coef_[0]
                else:
                    slopes[group] = 0
            
            # Tạo cột giả định để kiểm tra hiệu quả
            # Công thức: Giá trị gốc * Slope của nhóm
            temp_col = df[col] * df[key].map(slopes).fillna(0)
            temp_corr = temp_col.corr(df[target_col])
            
            # Ghi nhận nếu kết quả tốt hơn kỷ lục hiện tại
            if abs(temp_corr) > best_corr:
                best_corr = abs(temp_corr)
                best_key = key
                best_new_col_name = f"{col}_x_{key}_Optimized"
                best_slope_map = slopes

        # 3. Áp dụng thay đổi tốt nhất (nếu có cải thiện đáng kể > 0.01)
        if best_key and (best_corr > current_corrs[col] + 0.01):
            print(f"  [FIX] {col}: Tương quan {current_corrs[col]:.3f} -> {best_corr:.3f} (Ghép với '{best_key}')")
            
            # Tạo cột mới chính thức trong DataFrame
            df[best_new_col_name] = df[col] * df[best_key].map(best_slope_map).fillna(0)
            
            results_log.append({
                'Feature': col,
                'Optimized_By': best_key,
                'New_Corr': best_corr
            })
            
            # Đánh dấu xóa cột cũ để tránh trùng lặp thông tin
            if col not in cols_to_drop:
                cols_to_drop.append(col)
        else:
            print(f"  [SKIP] {col}: Không tìm thấy cách ghép nào hiệu quả (Max {best_corr:.3f}). Có thể do dữ liệu nhiễu hoàn toàn.")

    # 4. Xóa cột cũ và Lưu file
    if cols_to_drop:
        print(f"\nĐang xóa {len(cols_to_drop)} cột gốc yếu...")
        df.drop(columns=cols_to_drop, inplace=True)
    
    # Đặt tên file đầu ra
    output_file = 'Agri_Data_Final_Ready_For_Model1.csv'
    df.to_csv(output_file, index=False)
    print(f"\n--- HOÀN TẤT! Đã lưu file tối ưu tại: {output_file} ---")
    print(f"Tổng số cột hiện tại: {df.shape[1]}")

# --- CHẠY HÀM ---
# Lưu ý: Đảm bảo file 'Agri_Data_Cleaned_Processed.csv' nằm cùng thư mục
super_optimize_features('Agri_Data_Final_Ready_For_Model.csv')

--- BẮT ĐẦU TỐI ƯU HÓA SÂU: Agri_Data_Final_Ready_For_Model.csv ---
Phát hiện 0 cột yếu cần xử lý lại: []

--- HOÀN TẤT! Đã lưu file tối ưu tại: Agri_Data_Final_Ready_For_Model1.csv ---
Tổng số cột hiện tại: 43


In [44]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Đọc dữ liệu
file_path = 'Agri_Data_Final_Ready_For_Model1.csv'
df = pd.read_csv(file_path)

# 2. Tách dữ liệu
# test_size=0.2: Tách 20% cho tập kiểm tra (Test), 80% còn lại cho tập huấn luyện (Train)
# stratify=df['Crop Name']: Đảm bảo mỗi loại cây trồng đều xuất hiện trong cả 2 tập theo đúng tỉ lệ
train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['Crop Name']
)

# 3. Lưu thành 2 file riêng biệt
train_df.to_csv('train_data_80.csv', index=False)
test_df.to_csv('test_data_20.csv', index=False)

print(f"Đã tách xong!")
print(f"- File Train (80%): {len(train_df)} dòng")
print(f"- File Test (20%): {len(test_df)} dòng")

Đã tách xong!
- File Train (80%): 3341 dòng
- File Test (20%): 836 dòng


In [45]:
import pandas as pd

# 1. Đọc dữ liệu (đã tách trước đó)
train_df = pd.read_csv('train_data_80.csv')
test_df = pd.read_csv('test_data_20.csv')

# 2. Định nghĩa các nhóm cột cần xử lý
low_cardinality_cols = ['Season', 'pH_Suitability', 'Dominant_Soil_Texture', 'Water_Availability_Cat', 'Extreme_Heat_Risk']
high_cardinality_cols = ['District', 'Crop Name', 'Transplant', 'Growth', 'Harvest']

# ==========================================
# BƯỚC 3: TARGET ENCODING (Tránh Data Leakage)
# ==========================================
# Nguyên tắc: Chỉ tính toán thống kê trên tập TRAIN, sau đó áp dụng cho cả TRAIN và TEST.

# Tính giá trị trung bình chung của Yield trên tập Train (dùng để điền cho các giá trị lạ ở tập Test nếu có)
global_mean_yield = train_df['Yield'].mean()

for col in high_cardinality_cols:
    # Tính bảng map từ tập TRAIN
    target_mean_map = train_df.groupby(col)['Yield'].mean()
    
    # Áp dụng map cho tập TRAIN
    train_df[col + '_Target_Encoded'] = train_df[col].map(target_mean_map)
    
    # Áp dụng map đó cho tập TEST
    test_df[col + '_Target_Encoded'] = test_df[col].map(target_mean_map)
    
    # Xử lý trường hợp tập Test có category lạ (chưa từng xuất hiện ở Train) -> Điền bằng global mean
    test_df[col + '_Target_Encoded'].fillna(global_mean_yield, inplace=True)
    
    # Xóa cột gốc dạng chữ
    train_df.drop(columns=[col], inplace=True)
    test_df.drop(columns=[col], inplace=True)

# ==========================================
# BƯỚC 4: ONE-HOT ENCODING & ĐỒNG BỘ CỘT
# ==========================================
# Áp dụng get_dummies cho cả 2 tập
train_df = pd.get_dummies(train_df, columns=low_cardinality_cols, drop_first=True)
test_df = pd.get_dummies(test_df, columns=low_cardinality_cols, drop_first=True)

# QUAN TRỌNG: Đảm bảo tập Test có đúng và đủ các cột như tập Train
# Lấy danh sách cột chuẩn từ tập Train
train_cols = train_df.columns

# Thêm các cột thiếu vào Test (điền 0) và bỏ các cột thừa (nếu có)
test_df = test_df.reindex(columns=train_cols, fill_value=0)

# ==========================================
# BƯỚC 5: TÁCH X VÀ y
# ==========================================
# Tách tập Train
y_train = train_df['Yield']
X_train = train_df.drop(columns=['Yield'])

# Tách tập Test
y_test = test_df['Yield']
X_test = test_df.drop(columns=['Yield'])

# ==========================================
# BƯỚC 6: LƯU FILE
# ==========================================
X_train.to_csv('X_train.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("Đã hoàn tất xử lý!")
print(f"Kích thước X_train: {X_train.shape}")
print(f"Kích thước X_test: {X_test.shape}")

Đã hoàn tất xử lý!
Kích thước X_train: (3341, 44)
Kích thước X_test: (836, 44)


C:\Users\AKANEMO\AppData\Local\Temp\ipykernel_11760\1773555210.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_df[col + '_Target_Encoded'].fillna(global_mean_yield, inplace=True)
C:\Users\AKANEMO\AppData\Local\Temp\ipykernel_11760\1773555210.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alwa